**!!! WORK IN PROGRESS !!!**

In [ ]:

import os
import time
import pandas

# LOCALS:
from config import SITES
from utils import save_dataframe_as_csv
from hubeau_data_collectors import request_stations, download_hourly_water_heights, download_daily_flow_rates

In [ ]:
def main(stations_names, local_output_directory):
    os.makedirs(local_output_directory, exist_ok=True)
    stations_data_filepath = os.path.join(local_output_directory, "stations.csv")

    print(f"\n{'='*60}")
    print("Collecte des données hydrométriques sur Hub'Eau")
    print(f"Dossier de sortie : {local_output_directory}/")
    print(f"{'='*60}")

    # --- 1. Recherche des codes de stations ---
    stations_data = request_stations(stations_names)

    # Sauvegarder le référentiel des stations
    ref = []
    for nom, info in stations_data.items():
        if info:
            ref.append({"station": nom, **info, **SITES[nom]})
    
    pandas.DataFrame(ref).to_csv(stations_data_filepath, index=False)

    # --- 2. Téléchargement des données ---
    fichiers = []

    for nom, cfg in SITES.items():
        info = stations_data.get(nom)
        if not info:
            print(f"\n⚠ Station {nom} non trouvée, ignorée.")
            continue

        code = info["code"]
        debut, fin = cfg["periode"]

        print(f"\n{'='*60}")
        print(f"Station : {nom} | Code : {code} | Cours d'eau : {info['cours_eau']}")

        # 2a. Hauteurs horaires (données brutes "temps réel")
        df_h = download_hourly_water_heights(code, nom, debut, fin)
        f = save_dataframe_as_csv(df_h, nom, "hourly_water_heights", output_dir=local_output_directory)
        if f:
            fichiers.append(f)

        time.sleep(1)

        # 2b. Débits moyens journaliers (données élaborées, historique complet)
        df_q = download_daily_flow_rates(code, nom, debut, fin)
        f = save_dataframe_as_csv(df_q, nom, "daily_flow_rates", output_dir=local_output_directory)
        if f:
            fichiers.append(f)

        time.sleep(1)

    # --- Récapitulatif ---
    print(f"\n{'='*60}")
    print(f"✅ Collecte terminée — {len(fichiers)} fichier(s) créé(s) :")
    for f in fichiers:
        print(f"   {f}")

    final_msg = """
    ⚠  RAPPEL SUR LES DONNÉES "VÉRIFIÉES" :
    L'article utilise les données statut=16 ("vérifiées"), disponibles
    uniquement via l'interface web HydroPortail :
    → https://hydro.eaufrance.fr
    → Échanges → Exports hydrométriques → Séries de mesures
    (compte gratuit requis)

    Les données récupérées ici via l'API sont des données "temps réel"
    (brutes), qui peuvent différer légèrement des données vérifiées.
    """
    print(final_msg)

    return None
    

In [ ]:
main()  # TODO : args

TypeError: main() missing 2 required positional arguments: 'stations_names' and 'local_output_directory'